# Mammo-GRG four-view report generation

This demo loads four precomputed Mammo-FM feature maps in the training view order, then generates one report with the released stage-2 Mammo-GRG checkpoint. Run the notebook from the `Mammo-GRG` directory or its `demo` directory. Keep checkpoints and feature maps outside the source repository.

In [ ]:
import os
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "demo":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from mammo_grg.inference import (
    generate_report,
    load_four_view_embeddings,
    load_model,
)

## Configure the checkpoint and four feature maps

Each `.pt` file must be the raw Mammo-FM feature map for one view, with shape `[1, 2048, 48, 29]`.

In [ ]:
ASSET_ROOT = Path(os.environ.get("MAMMO_GRG_ASSET_DIR", "/path/to/Mammo-FM-Test-check/Mammo-GRG"))
MODEL_PATH = ASSET_ROOT / "stage2/checkpoint-5500"
MODEL_BASE = "meta-llama/Meta-Llama-3.1-8B-Instruct"
DEVICE = "cuda"
EMBEDDING_ROOT = Path("/path/to/four-view-embeddings")

VIEW_EMBEDDINGS = {
    "LMLO": EMBEDDING_ROOT / "LMLO.pt",
    "LCC": EMBEDDING_ROOT / "LCC.pt",
    "RMLO": EMBEDDING_ROOT / "RMLO.pt",
    "RCC": EMBEDDING_ROOT / "RCC.pt",
}
PROMPT = "Generate a complete mammography report for this four-view examination."

## Load Mammo-GRG

In [ ]:
tokenizer, model = load_model(
    model_path=MODEL_PATH,
    model_base=MODEL_BASE,
    device=DEVICE,
    load_4bit=False,
)
print(f"Loaded {MODEL_PATH}")

## Load the four precomputed embeddings

In [ ]:
view_order = ("LMLO", "LCC", "RMLO", "RCC")
embeddings = load_four_view_embeddings(
    [VIEW_EMBEDDINGS[view] for view in view_order],
    device=model.device,
    dtype=model.dtype,
)
print(embeddings.shape)  # [1, 4, 1, 2048, 48, 29]

## Generate the report

In [ ]:
report = generate_report(
    model=model,
    tokenizer=tokenizer,
    embeddings=embeddings,
    question=PROMPT,
    max_new_tokens=512,
    temperature=0.0,
)
print(report)